# K1 Humanoid — Control Notebook
**Booster K1 · Jetson Orin NX · ROS2 · AI Innovation Center, Hillsborough Community College**

---

## How to use this notebook

| Cell | Purpose |
|------|---------|
| **Cell 1** | Connect — SSH into the K1 Jetson over Ethernet or hotspot |
| **Cell 2** | Startup — DAMP → PREP → WALK safety sequence |
| **Cell 3** | Dashboard — interactive controls, sensors, AI chat |
| **Cell 4** | Shutdown — safe power-down and disconnect |

---

### ⚠️ Safety rules — read before running

1. **Never skip the PREP hold.** After PREP, wait a full **3 seconds** before sending any motion command. Skipping this can damage joints.
2. **Clear the area before WALK.** K1 is full-size. Give it at least 1.5m clearance in all directions.
3. **Two-battery system.** Each session requires two charged batteries. Check both before starting.
4. **DAMP first, always.** If something goes wrong, send DAMP immediately to cut motor power.
5. **Keep Jetson clean.** Do not run LLM inference or heavy computation on the Jetson. All AI runs on the operator laptop.

---

### Network options

| Connection | K1 IP | Use when |
|------------|-------|----------|
| Direct Ethernet | `192.168.10.102` | Most reliable; use for demos |
| Franklin T10 Hotspot | `192.168.0.176` | Wireless; use for movement demos |

SSH credentials: **username** `booster` · **password** `123456`

## Cell 1 — Connect to K1

In [ ]:
"""
K1 Control Notebook — Cell 1: Connect
Booster K1 · Jetson Orin NX · AI Innovation Center, HCC

Establishes SSH connection to the K1 Jetson.
Run this cell first every session.
"""

import paramiko
import subprocess
import time
import threading
import json
import urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── EDIT THESE IF NEEDED ─────────────────────────────────────────────────────
K1_IP       = '192.168.10.102'   # Direct Ethernet (default)
# K1_IP     = '192.168.0.176'    # Franklin T10 Hotspot (uncomment if wireless)
K1_USER     = 'booster'
K1_PASSWORD = '123456'
OLLAMA_URL  = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'llama3.2'
# ─────────────────────────────────────────────────────────────────────────────

client = None

def k1_connect():
    """Open SSH connection to the K1 Jetson."""
    global client
    try:
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        client.connect(
            K1_IP,
            username=K1_USER,
            password=K1_PASSWORD,
            timeout=10
        )
        print(f'✅ Connected to K1 at {K1_IP}')
        print(f'   User: {K1_USER}')
        return True
    except Exception as e:
        print(f'❌ Connection failed: {e}')
        print('   Check: Is K1 powered on? Is your laptop on the same network?')
        client = None
        return False

def k1_do(command, timeout=30):
    """
    Send a shell command to the K1 Jetson over SSH.
    Returns stdout as a string.
    
    This is the core function used by every cell in this notebook.
    Every robot action is a shell command piped through this SSH session.
    """
    global client
    if client is None or not client.get_transport() or not client.get_transport().is_active():
        print('Not connected. Run Cell 1 again.')
        return None
    try:
        chan = client.get_transport().open_session()
        chan.settimeout(timeout)
        chan.exec_command(command)
        out = chan.makefile('r', -1).read().strip()
        err = chan.makefile_stderr('r', -1).read().strip()
        chan.close()
        return out if out else ('ERROR: ' + err if err else 'OK')
    except Exception as e:
        return f'Exception: {e}'

def check_battery():
    """Read battery status from K1. Returns raw output string."""
    result = k1_do('python3 /home/booster/Workspace/Desh/battery_monitor.py --once')
    return result

def k1_speak(text):
    """
    Send text to K1's speaker using espeak-ng.
    NOTE: espeak-ng must be installed on the Jetson:
        sudo apt-get install -y espeak-ng
    """
    safe = text.replace('"', '').replace("'", '')[:300]
    cmd = (
        f'espeak-ng "{safe}" --stdout | '
        f'paplay --device=alsa_output.usb-C-Media_Electronics_Inc.'
        f'_USB_Audio_Device-00.analog-stereo'
    )
    return k1_do(cmd, timeout=30)

# ── Run connection ────────────────────────────────────────────────────────────
k1_connect()

# Quick connectivity test
ping = k1_do('echo K1 ready')
if ping:
    print(f'   Jetson response: {ping}')
    batt = check_battery()
    print(f'   Battery status: {batt}')


## Cell 2 — Startup: DAMP → PREP → WALK

> **Run this cell only after Cell 1 confirms connection.**  
> The PREP phase enforces a 3-second hold automatically. Do not interrupt.

In [ ]:
"""
K1 Control Notebook — Cell 2: Startup Sequence

Executes the mandatory DAMP → PREP → WALK startup sequence.
This sequence must be followed every session to protect the robot's joints.

DAMP  = motors unpowered (default state at power-on)
PREP  = motors engage; robot stands and stiffens joints (3-second hold required)
WALK  = robot ready for locomotion and gesture commands
"""

import time

PREP_HOLD_SECONDS = 3   # Do not reduce this value

def startup_sequence():
    print('── K1 Startup Sequence ──────────────────────────────')
    print()

    # ── DAMP ─────────────────────────────────────────────────────────────────
    print('⚫ DAMP  — Confirming motors are unpowered...')
    result = k1_do(
        'python3 -c "'
        'from booster_sdk import BoosterRobot; '
        'r = BoosterRobot(); '
        'r.set_mode(\"damp\"); '
        'print(\"DAMP OK\")"',
        timeout=15
    )
    print(f'   {result}')
    time.sleep(1)

    # ── PREP ─────────────────────────────────────────────────────────────────
    print()
    print('🟡 PREP  — Engaging motors and standing K1...')
    print('   ⚠️  Stand clear. K1 is standing up.')
    result = k1_do(
        'python3 -c "'
        'from booster_sdk import BoosterRobot; '
        'r = BoosterRobot(); '
        'r.set_mode(\"prep\"); '
        'print(\"PREP OK\")"',
        timeout=20
    )
    print(f'   {result}')

    # Mandatory 3-second hold — do not skip
    print(f'   Holding PREP for {PREP_HOLD_SECONDS} seconds...')
    for i in range(PREP_HOLD_SECONDS, 0, -1):
        print(f'   {i}...')
        time.sleep(1)
    print('   PREP hold complete.')

    # ── WALK ─────────────────────────────────────────────────────────────────
    print()
    print('🟢 WALK  — K1 ready for commands.')
    result = k1_do(
        'python3 -c "'
        'from booster_sdk import BoosterRobot; '
        'r = BoosterRobot(); '
        'r.set_mode(\"walk\"); '
        'print(\"WALK OK\")"',
        timeout=20
    )
    print(f'   {result}')

    print()
    print('── Startup complete. Run Cell 3 for the dashboard. ──')

startup_sequence()


## Cell 3 — Interactive Dashboard

Run this cell after the startup sequence completes successfully.  
The dashboard provides controls for motion, speech, sensors, and AI chat.

> **DAMP button is always available.** Click it immediately if K1 behaves unexpectedly.

In [ ]:
"""
K1 Control Notebook — Cell 3: Interactive Dashboard

Sections:
  A. Mode Control   — DAMP / PREP / WALK
  B. Motion         — forward, back, turn, stop
  C. Gestures       — wave, nod, bow
  D. Speech         — speak custom text
  E. Sensors        — battery, sonar, joint states
  F. AI Chat        — Ollama/llama3.2 conversation logged live
"""

import ipywidgets as widgets
from IPython.display import display, clear_output
import time, threading, json, urllib.request

out = widgets.Output()
chat_log = []

def log(msg):
    with out:
        print(msg)

# ── SECTION A: Mode Control ───────────────────────────────────────────────────
lbl_mode = widgets.HTML('<b style="font-size:14px">⚙️ Mode Control</b>')
btn_damp = widgets.Button(description='⚫ DAMP',  button_style='danger',  layout=widgets.Layout(width='120px'))
btn_prep = widgets.Button(description='🟡 PREP',  button_style='warning', layout=widgets.Layout(width='120px'))
btn_walk = widgets.Button(description='🟢 WALK',  button_style='success', layout=widgets.Layout(width='120px'))

def set_mode(mode):
    log(f'Setting mode: {mode.upper()}...')
    if mode == 'prep':
        log('  Holding PREP for 3 seconds...')
    cmd = (
        f'python3 -c "from booster_sdk import BoosterRobot; '
        f'r = BoosterRobot(); r.set_mode(\\"{mode}\\"); print(\\"{mode.upper()} OK\\")"'
    )
    result = k1_do(cmd, timeout=25)
    if mode == 'prep':
        time.sleep(3)
    log(f'  {result}')

btn_damp.on_click(lambda b: threading.Thread(target=set_mode, args=('damp',)).start())
btn_prep.on_click(lambda b: threading.Thread(target=set_mode, args=('prep',)).start())
btn_walk.on_click(lambda b: threading.Thread(target=set_mode, args=('walk',)).start())

row_mode = widgets.HBox([btn_damp, btn_prep, btn_walk])

# ── SECTION B: Motion ─────────────────────────────────────────────────────────
lbl_motion = widgets.HTML('<b style="font-size:14px">🚶 Motion</b>')
spd = widgets.FloatSlider(value=0.3, min=0.1, max=1.0, step=0.1,
    description='Speed:', style={'description_width': '60px'},
    layout=widgets.Layout(width='300px'))

btn_fwd  = widgets.Button(description='▲ Forward', layout=widgets.Layout(width='110px'))
btn_back = widgets.Button(description='▼ Back',    layout=widgets.Layout(width='110px'))
btn_left = widgets.Button(description='◀ Turn L',  layout=widgets.Layout(width='110px'))
btn_right= widgets.Button(description='▶ Turn R',  layout=widgets.Layout(width='110px'))
btn_stop = widgets.Button(description='■ Stop',    button_style='danger', layout=widgets.Layout(width='110px'))

def move(vx=0.0, vy=0.0, vtheta=0.0):
    s = spd.value
    cmd = (
        f'python3 -c "from booster_sdk import BoosterRobot; '
        f'r = BoosterRobot(); r.move({vx*s:.2f}, {vy:.2f}, {vtheta:.2f}); '
        f'print(\\"Move OK\\")"'
    )
    result = k1_do(cmd, timeout=10)
    log(f'Motion: vx={vx*s:.2f} vtheta={vtheta:.2f} → {result}')

btn_fwd.on_click(lambda b:  threading.Thread(target=move, kwargs={'vx': 1.0}).start())
btn_back.on_click(lambda b: threading.Thread(target=move, kwargs={'vx': -1.0}).start())
btn_left.on_click(lambda b: threading.Thread(target=move, kwargs={'vtheta': 1.0}).start())
btn_right.on_click(lambda b:threading.Thread(target=move, kwargs={'vtheta': -1.0}).start())
btn_stop.on_click(lambda b: threading.Thread(target=move).start())

row_motion1 = widgets.HBox([btn_fwd, btn_back, btn_left, btn_right, btn_stop])

# ── SECTION C: Gestures ──────────────────────────────────────────────────────
lbl_gest = widgets.HTML('<b style="font-size:14px">🤖 Gestures</b>')
btn_wave = widgets.Button(description='Wave',   layout=widgets.Layout(width='100px'))
btn_nod  = widgets.Button(description='Nod',    layout=widgets.Layout(width='100px'))
btn_bow  = widgets.Button(description='Bow',    layout=widgets.Layout(width='100px'))
btn_neutral = widgets.Button(description='Neutral', layout=widgets.Layout(width='100px'))

GESTURES = {
    'wave':    'r.do_gesture(\\"wave\\")',
    'nod':     'r.do_gesture(\\"nod\\")',
    'bow':     'r.do_gesture(\\"bow\\")',
    'neutral': 'r.stand()',
}

def gesture(name):
    action = GESTURES.get(name, 'r.stand()')
    cmd = f'python3 -c "from booster_sdk import BoosterRobot; r = BoosterRobot(); {action}; print(\\"{name} OK\\")"'
    result = k1_do(cmd, timeout=20)
    log(f'Gesture {name}: {result}')

btn_wave.on_click(lambda b:    threading.Thread(target=gesture, args=('wave',)).start())
btn_nod.on_click(lambda b:     threading.Thread(target=gesture, args=('nod',)).start())
btn_bow.on_click(lambda b:     threading.Thread(target=gesture, args=('bow',)).start())
btn_neutral.on_click(lambda b: threading.Thread(target=gesture, args=('neutral',)).start())

row_gest = widgets.HBox([btn_wave, btn_nod, btn_bow, btn_neutral])

# ── SECTION D: Speech ─────────────────────────────────────────────────────────
lbl_speech = widgets.HTML('<b style="font-size:14px">🔊 Speech</b>')
txt_speak  = widgets.Text(placeholder='Type something for K1 to say...',
                          layout=widgets.Layout(width='500px'))
btn_speak  = widgets.Button(description='Speak', button_style='info',
                            layout=widgets.Layout(width='100px'))

def do_speak(b):
    text = txt_speak.value.strip()
    if not text:
        return
    log(f'Speaking: "{text}"')
    result = k1_speak(text)
    log(f'  → {result}')

btn_speak.on_click(lambda b: threading.Thread(target=do_speak, args=(b,)).start())
row_speech = widgets.HBox([txt_speak, btn_speak])

# ── SECTION E: Sensors ────────────────────────────────────────────────────────
lbl_sensors = widgets.HTML('<b style="font-size:14px">📡 Sensors</b>')
btn_battery  = widgets.Button(description='🔋 Battery',    layout=widgets.Layout(width='130px'))
btn_joints   = widgets.Button(description='🦾 Joint States', layout=widgets.Layout(width='130px'))
btn_camera   = widgets.Button(description='📷 Camera Info', layout=widgets.Layout(width='130px'))

def read_battery(b):
    log('Reading battery...')
    result = k1_do('python3 /home/booster/Workspace/Desh/battery_monitor.py --once', timeout=10)
    log(f'  Battery: {result}')

def read_joints(b):
    log('Reading joint states from ROS2 topic...')
    result = k1_do(
        'timeout 3 ros2 topic echo /joint_states --once 2>/dev/null || echo "No joint state topic found"',
        timeout=10
    )
    log(f'  {result[:500]}')

def read_camera(b):
    log('Reading ZED camera info...')
    result = k1_do(
        'timeout 3 ros2 topic list | grep -i image || echo "No image topics found"',
        timeout=10
    )
    log(f'  Camera topics: {result}')

btn_battery.on_click(lambda b: threading.Thread(target=read_battery, args=(b,)).start())
btn_joints.on_click(lambda b:  threading.Thread(target=read_joints,  args=(b,)).start())
btn_camera.on_click(lambda b:  threading.Thread(target=read_camera,  args=(b,)).start())

row_sensors = widgets.HBox([btn_battery, btn_joints, btn_camera])

# ── SECTION F: AI Chat ────────────────────────────────────────────────────────
lbl_ai   = widgets.HTML('<b style="font-size:14px">🧠 AI Chat (Ollama / llama3.2)</b>')
txt_chat = widgets.Text(placeholder='Ask K1 anything...',
                        layout=widgets.Layout(width='500px'))
btn_chat = widgets.Button(description='Send', button_style='primary',
                          layout=widgets.Layout(width='100px'))
lbl_pipeline = widgets.HTML(
    '<small style="color:#666">Pipeline: text → Ollama (laptop) → K1 speaker (espeak-ng)</small>'
)

K1_PERSONA = (
    'You are K1, a full-size humanoid robot at the Hillsborough Community College '
    'AI Innovation Center. You are powerful, precise, and focused. '
    'Respond in 1-2 short, confident sentences.'
)

def do_chat(b):
    text = txt_chat.value.strip()
    if not text:
        return
    chat_log.append({'role': 'user', 'content': text})
    log(f'You: {text}')
    try:
        payload = json.dumps({
            'model': OLLAMA_MODEL,
            'prompt': f'{K1_PERSONA}\n\nUser: {text}',
            'stream': False
        }).encode()
        req = urllib.request.Request(
            OLLAMA_URL, data=payload,
            headers={'Content-Type': 'application/json'}
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            reply = json.loads(resp.read())['response'].strip()
        chat_log.append({'role': 'assistant', 'content': reply})
        log(f'K1: {reply}')
        k1_speak(reply)
        txt_chat.value = ''
    except Exception as e:
        log(f'AI error: {e}')

btn_chat.on_click(lambda b: threading.Thread(target=do_chat, args=(b,)).start())
row_chat = widgets.HBox([txt_chat, btn_chat])

# ── ASSEMBLE DASHBOARD ────────────────────────────────────────────────────────
dashboard = widgets.VBox([
    widgets.HTML('<hr><h2 style="color:#1A5E8C">🤖 K1 Control Center</h2>'),
    widgets.HTML(f'<small>Connected to: <b>{K1_IP}</b> · Model: <b>{OLLAMA_MODEL}</b></small>'),
    widgets.HTML('<hr>'),
    lbl_mode,   row_mode,   widgets.HTML('<hr>'),
    lbl_motion, spd, row_motion1, widgets.HTML('<hr>'),
    lbl_gest,   row_gest,   widgets.HTML('<hr>'),
    lbl_speech, row_speech, widgets.HTML('<hr>'),
    lbl_sensors,row_sensors,widgets.HTML('<hr>'),
    lbl_ai, lbl_pipeline, row_chat, widgets.HTML('<hr>'),
    widgets.HTML('<b>📋 Log</b>'),
    out
])

display(dashboard)
log('K1 Control Center ready. Use the buttons above.')


## Cell 4 — Shutdown

> **Always run the full sequence. Never go directly to DAMP from WALK or PREP.**
>
> Cutting motor power while K1 is standing causes an uncontrolled fall and joint damage.
>
> **Sequence: Sit → Lie Down → DAMP → Disconnect**

| Step | Action | Why |
|------|--------|-----|
| 1 | **Sit** | Lowers center of gravity safely |
| 2 | **Lie Down** | Brings K1 fully to ground level |
| 3 | **DAMP** | Cuts motor power from a safe position |
| 4 | **Disconnect** | Closes SSH session cleanly |

In [ ]:
"""
K1 Control Notebook — Cell 4: Shutdown

Safe shutdown sequence:
  1. Sit     — K1 lowers into a seated position
  2. Lie Down — K1 moves to a prone/rest position on the ground
  3. DAMP    — motors power off safely from ground level
  4. SSH close

NEVER go directly from WALK or PREP to DAMP.
Cutting motor power while standing causes K1 to fall and damages joints.
Always run this full sequence before powering off.
"""

import time

def shutdown():
    global client
    print('── K1 Shutdown ─────────────────────────────────────')
    print()
    print('⚠️  Stand clear. K1 is beginning shutdown sequence.')
    print()

    # ── Step 1: Sit ──────────────────────────────────────────────────────────
    print('🪑 Step 1: Sit — K1 lowering to seated position...')
    result = k1_do(
        'python3 -c "from booster_sdk import BoosterRobot; '
        'r = BoosterRobot(); r.do_gesture(\\"sit\\"); '
        'print(\\"Sit OK\\")"  ',
        timeout=20
    )
    print(f'   {result}')
    print('   Waiting for sit to complete...')
    time.sleep(4)   # Allow full sit motion to complete

    # ── Step 2: Lie Down ─────────────────────────────────────────────────────
    print()
    print('🛌 Step 2: Lie Down — K1 moving to rest position...')
    result = k1_do(
        'python3 -c "from booster_sdk import BoosterRobot; '
        'r = BoosterRobot(); r.do_gesture(\\"lie_down\\"); '
        'print(\\"Lie Down OK\\")"  ',
        timeout=20
    )
    print(f'   {result}')
    print('   Waiting for lie-down to complete...')
    time.sleep(4)   # Allow full lie-down motion to complete

    # ── Step 3: DAMP ─────────────────────────────────────────────────────────
    print()
    print('⚫ Step 3: DAMP — cutting motor power from ground level...')
    result = k1_do(
        'python3 -c "from booster_sdk import BoosterRobot; '
        'r = BoosterRobot(); r.set_mode(\\"damp\\"); '
        'print(\\"DAMP OK\\")"  ',
        timeout=15
    )
    print(f'   {result}')
    time.sleep(1)

    # ── Step 4: Close SSH ────────────────────────────────────────────────────
    print()
    print('🔌 Step 4: Closing SSH connection...')
    try:
        if client:
            client.close()
            client = None
        print('   SSH closed.')
    except Exception as e:
        print(f'   Error closing SSH: {e}')

    print()
    print('── Shutdown complete. ───────────────────────────────')
    print('   K1 is on the ground with motors off.')
    print('   Safe to power off. Remove both batteries after shutdown.')

shutdown()


---
## Extension Cells — Add Your Own

Use these starter cells for Phase 3 activities. Add new cells below as needed.

| Extension | What you'll learn |
|-----------|-------------------|
| **Ext A** | Subscribe to a ROS2 topic and log joint positions |
| **Ext B** | Battery threshold alert — speak when battery drops below 20% |
| **Ext C** | Voice-triggered gesture — use Whisper STT to trigger K1 actions by voice |
| **Ext D** | Sim-to-real — run a joint command in Isaac Sim, then mirror it on the physical K1 |

In [ ]:
# ── Extension A: ROS2 Joint State Subscriber ─────────────────────────────────
# Goal: subscribe to /joint_states and log position data for each joint.
#
# Library to explore: rclpy
# Ask AI: "I'm using rclpy in Python. Show me how to subscribe to a
#          /joint_states topic and print the position array each time
#          a message arrives."
#
# Then: verify the method names against the rclpy docs before running.

# Write your subscriber here:
# ─────────────────────────────────────────────────────────────────────────────

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState

class JointLogger(Node):
    def __init__(self):
        super().__init__('joint_logger')
        self.sub = self.create_subscription(
            JointState,
            '/joint_states',
            self.callback,
            10
        )

    def callback(self, msg):
        # TODO: log msg.name and msg.position to a list or CSV
        print(f'Joints: {list(zip(msg.name, msg.position))}')

# rclpy.init()
# node = JointLogger()
# rclpy.spin_once(node, timeout_sec=3.0)
# node.destroy_node()
print('Extension A starter loaded. Uncomment the last 4 lines to run.')


In [ ]:
# ── Extension B: Battery Threshold Alert ──────────────────────────────────────
# Goal: poll the battery every 60 seconds. If charge drops below 20%,
#       make K1 speak an alert.
#
# Ask AI: "I have a function k1_do() that runs SSH commands on a robot
#          and a function k1_speak(text) that makes the robot talk.
#          Show me a Python loop that checks battery every 60 seconds
#          and calls k1_speak() if battery is below 20."

import time

BATTERY_THRESHOLD = 20   # percent
POLL_INTERVAL     = 60   # seconds

def battery_monitor_loop(cycles=5):
    """Poll battery for `cycles` iterations. Modify or extend as needed."""
    for i in range(cycles):
        raw = k1_do('python3 /home/booster/Workspace/Desh/battery_monitor.py --once', timeout=10)
        print(f'[{i+1}/{cycles}] Battery: {raw}')

        # TODO: parse raw to extract integer percentage
        # TODO: if pct < BATTERY_THRESHOLD: k1_speak('Battery low. Please recharge.')

        if i < cycles - 1:
            time.sleep(POLL_INTERVAL)

# battery_monitor_loop(cycles=3)
print('Extension B starter loaded. Uncomment the last line to run.')


In [ ]:
# ── Extension C: Voice-Triggered Gesture ──────────────────────────────────────
# Goal: use Whisper STT to listen for spoken keywords and trigger K1 gestures.
#       Say 'wave' → K1 waves. Say 'bow' → K1 bows.
#
# Ask AI: "I'm using OpenAI Whisper in Python to transcribe audio from a
#          microphone. Show me how to record 3 seconds of audio using
#          sounddevice and pass it to whisper.load_model().transcribe()."

import whisper
import sounddevice as sd
import numpy as np

KEYWORD_MAP = {
    'wave':    'r.do_gesture(\"wave\")',
    'bow':     'r.do_gesture(\"bow\")',
    'nod':     'r.do_gesture(\"nod\")',
    'stop':    'r.move(0, 0, 0)',
}

def listen_and_act(duration=3, fs=16000):
    print(f'Listening for {duration} seconds...')
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()
    audio_np = audio.flatten()

    model = whisper.load_model('tiny')
    result = model.transcribe(audio_np)
    text = result['text'].strip().lower()
    print(f'Heard: "{text}"')

    for keyword, action in KEYWORD_MAP.items():
        if keyword in text:
            print(f'Keyword matched: {keyword}')
            cmd = f'python3 -c "from booster_sdk import BoosterRobot; r = BoosterRobot(); {action}"'
            result = k1_do(cmd, timeout=20)
            print(f'Result: {result}')
            return
    print('No keyword matched.')

# listen_and_act()
print('Extension C starter loaded. Uncomment the last line to run.')


In [ ]:
# ── Extension D: Sim-to-Real Joint Command ────────────────────────────────────
# Goal: define a joint angle target, run it in Isaac Sim first,
#       then send the identical command to the physical K1.
#
# Ask AI: "I'm using omni.isaac.kit and UsdPhysics.DriveAPI to control
#          a robot in Isaac Sim. Show me how to set a target position
#          on a single revolute joint by path."

# Joint angle targets (radians) — modify these values
TARGET_ANGLES = {
    'left_shoulder':  0.3,
    'right_shoulder': -0.3,
    'head_pitch':     0.0,
}

def sim_joint_command(stage, joint_path, angle_rad):
    """
    Set a joint target in Isaac Sim.
    Requires: omni.isaac.kit, pxr.UsdPhysics
    """
    # from pxr import UsdPhysics
    # drive = UsdPhysics.DriveAPI.Get(stage.GetPrimAtPath(joint_path), 'angular')
    # drive.GetTargetPositionAttr().Set(angle_rad)
    print(f'[Sim] Set {joint_path} to {angle_rad:.2f} rad')

def physical_joint_command(joint_name, angle_rad):
    """Send the same angle to the physical K1 via ROS2."""
    cmd = (
        f'python3 -c "'
        f'import rclpy; from rclpy.node import Node; '
        f'from std_msgs.msg import Float64MultiArray; '
        f'rclpy.init(); n = Node(\'cmd\'); '
        f'p = n.create_publisher(Float64MultiArray, \'/joint_group_position_controller/commands\', 1); '
        f'm = Float64MultiArray(); m.data = [{angle_rad:.3f}]; p.publish(m); '
        f'print(\'Joint command sent\')"'
    )
    result = k1_do(cmd, timeout=10)
    print(f'[Physical] {joint_name} = {angle_rad:.2f} rad → {result}')

# Sim run (comment out if Isaac Sim not available):
# for joint, angle in TARGET_ANGLES.items():
#     sim_joint_command(stage, f'/K1/{joint}', angle)

# Physical run (uncomment when ready):
# for joint, angle in TARGET_ANGLES.items():
#     physical_joint_command(joint, angle)

print('Extension D starter loaded. Uncomment the sim or physical blocks to run.')
